In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Analysis 
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Preprocessing 
from sklearn.preprocessing import MinMaxScaler , StandardScaler ,LabelEncoder , OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

# process (Model)
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline

#Evaluation 
from sklearn.metrics import accuracy_score , r2_score ,mean_squared_error


import joblib

# Load datasets


In [ ]:

train = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/train.csv", parse_dates=['date'])
stores = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/stores.csv")
holidays = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv", parse_dates=['date'])
oil = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/oil.csv", parse_dates=['date'])
transactions = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/transactions.csv", parse_dates=['date'])



# Data samples

In [ ]:
train.sample()


In [ ]:
train.isnull().sum()

In [ ]:
stores.sample()

In [ ]:
stores.isnull().sum()

In [ ]:
holidays . sample()

In [ ]:
holidays.isnull().sum()

In [ ]:
transactions . sample()

In [ ]:
transactions.isnull().sum()

In [ ]:
oil . sample()

In [ ]:
oil.isnull().sum()

# Handling Missing Values

In [ ]:
# oil
oil["dcoilwtico"] = oil["dcoilwtico"].fillna(oil["dcoilwtico"].mean())


# Merge 

In [ ]:

df = train.merge(stores,  how="left")
df


In [ ]:
df = df.merge(holidays, on="date", how="left")
df

In [ ]:
df = df.merge(oil, on="date", how="left")
df

In [ ]:

df = df.merge(transactions, on=["date", "store_nbr"], how="left")

In [ ]:
df.describe()

In [ ]:
df.describe(include= "O")

# Handling Missing Values

In [ ]:

df.isnull().sum()

In [ ]:
df.shape

In [ ]:

holiday_cols = ["type_y", "locale", "locale_name", "description", "transferred"]
for col in holiday_cols:
    df[col] = df[col].fillna("No Holiday")


for col in df.select_dtypes(include="object").columns:
    if col not in holiday_cols:
        df[col] = df[col].fillna(df[col].mode()[0])


for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

# Dealing With Outliers 

In [ ]:
def remove_outliers_IQR(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[column] >= lower) & (df[column] <= upper)]
for col in df.select_dtypes(include=np.number).columns:
   df= remove_outliers_IQR(df,col)




In [ ]:
df.shape

In [ ]:
df.sample()

# Feature Engineering

In [ ]:
df["date"] = pd.to_datetime(df["date"])

In [ ]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day

In [ ]:
df["dayofweek"] = df["date"].dt.dayofweek  # 0=Mon .. 6=Sun

In [ ]:
# The weekend in Ecuadur is sat , sun
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

In [ ]:
df["is_holiday"] = (df["type_y"] != "No Holiday").astype(int)


In [ ]:

# (Cyclical Encoding)   to use it in linear regression

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

In [ ]:
df["type_x_encoded"] = df["type_x"].astype("category").cat.codes

In [ ]:
df["cluster"] = df["cluster"].astype(int)

In [ ]:
df["city_encoded"] = df["city"].astype("category").cat.codes
df["state_encoded"] = df["state"].astype("category").cat.codes


In [ ]:
df["onpromotion"] = df["onpromotion"].astype(int)

In [ ]:

df["transferred"] = pd.to_numeric(df["transferred"], errors="coerce")


df["transferred"] = df["transferred"].fillna(0).astype(int)


In [ ]:
df["dcoilwtico"] = pd.to_numeric(df["dcoilwtico"], errors="coerce")

In [ ]:
df["transactions"] = pd.to_numeric(df["transactions"], errors="coerce")

In [ ]:
df.isnull().sum()

# Droping unused col

In [ ]:
df = df.drop(columns=["family","city","state","type_x","type_y","locale","locale_name","description"])

# Scaling

In [ ]:
exclude_cols = ["id", "date", "sales"] 

numeric_features = df.select_dtypes(include=["int64", "float64"]).drop(columns=exclude_cols, errors="ignore").columns.tolist()
categorical_features = df.select_dtypes(include=["object", "category"]).drop(columns=exclude_cols, errors="ignore").columns.tolist()


In [ ]:

numeric_transformer=StandardScaler()
categorical_transformer=OneHotEncoder(handle_unknown="ignore")

preprocessor=ColumnTransformer(
    transformers=[("num",numeric_transformer,numeric_features),
                  ("cat",categorical_transformer,categorical_features)
                 ]
)

# Spiliting Data

In [ ]:
X = df.drop(columns=["sales", "date"])   
y = df["sales"]

  
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

# pipeline

In [ ]:
model= Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("regressor",LinearRegression())
]              
  )

In [ ]:
model.fit(X_train, y_train)

# Model Predection

In [ ]:
y_train_pred = model.predict(X_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
lr = model.named_steps["regressor"]

# Coefficients (weights) and intercept
weights = lr.coef_
bias = lr.intercept_

print("Bias (intercept):", bias)
print("Weights shape:", weights.shape)

In [ ]:
feature_names = preprocessor.get_feature_names_out()

# ابني DataFrame بال coefficients
coefficients = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": lr.coef_
}).sort_values(by="Coefficient", ascending=False)


plt.figure(figsize=(10,6))
plt.barh(coefficients["Feature"], coefficients["Coefficient"])
plt.xlabel("Coefficient Value")
plt.ylabel("Feature")
plt.title("Linear Regression Coefficients")
plt.gca().invert_yaxis() 
plt.show()

# Model Score

In [ ]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\nModel Evaluation:")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R² Score: {r2:.2f}")

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.6, color="blue")
plt.xlabel("Actual (y)")
plt.ylabel("Predicted (ŷ)")
plt.title("Linear Regression with Preprocessing")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red", linestyle="--")
plt.show()

# Saving

In [ ]:
joblib.dump(model, "linear_pipeline.joblib")

# load

In [ ]:
loaded_pipeline = joblib.load("linear_pipeline.joblib")
y_pred = loaded_pipeline.predict(X_test)